# Question 2: Model Comparison and Improvement

## Introduction

- This notebook addresses Question 2 of the NLP Project. Since Question 2 builds on the work completed in Question 1, selected code from Question 1 is repeated here to ensure that this notebook can run independently.

-The preprocessing, train-test split, and TF-IDF vectorisation steps are required to prepare the text data for training and evaluating the machine learning models in Question 2. These steps follow the same approach used in Question 1 to maintain consistency in the results.

- The remaining sections focus on comparing different machine learning models, improving model performance, and implementing the final model as required by Question 2.

In [1]:
import pandas as pd
import re

# load the data
df = pd.read_json("research_abstracts.json")

display(df.head())

print("\nDataset info")
df.info()

# check missing values
print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicats:")
print(df.duplicated().sum())

print("\nLabel counts:")
print(df["Label"].value_counts())

import nltk
from nltk.stem import WordNetLemmatizer

nltk.download("wordnet")

lemmatizer = WordNetLemmatizer()

# clean and reduce the words to their base form
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words).strip()

df["Clean_Abstract"] = df["Abstract"].apply(clean_text)

print("\nOriginal and cleaned text:")
display(df[["Abstract", "Clean_Abstract", "Label"]].head())

print("\nShape of the dataset:", df.shape)

,DOI,Abstract,Label
0,10.1155/2021/1071145,Forward-looking forecasting of the inflation r...,ai
1,10.1007/s43546-022-00384-2,Our study examined the disaggregation of infla...,ai
2,10.1002/for.2667,This paper develops a dynamic factor model tha...,classical
3,10.1007/s42973-021-00091-x,"In this paper, I use a simple SIR Macro model ...",classical
4,10.1007/s40258-012-0003-z,BackgroundGame theory is useful for identifyin...,classical



Dataset info
<class 'pandas.core.frame.DataFrame'>
Index: 1138 entries, 0 to 1137
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   DOI       1134 non-null   object
 1   Abstract  1138 non-null   object
 2   Label     1138 non-null   object
dtypes: object(3)
memory usage: 35.6+ KB

Missing values:
DOI         4
Abstract    0
Label       0
dtype: int64

Duplicats:
2

Label counts:
Label
classical    900
ai           238
Name: count, dtype: int64


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\xikon\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!



Original and cleaned text:


,Abstract,Clean_Abstract,Label
0,Forward-looking forecasting of the inflation r...,forward looking forecasting of the inflation r...,ai
1,Our study examined the disaggregation of infla...,our study examined the disaggregation of infla...,ai
2,This paper develops a dynamic factor model tha...,this paper develops a dynamic factor model tha...,classical
3,"In this paper, I use a simple SIR Macro model ...",in this paper i use a simple sir macro model t...,classical
4,BackgroundGame theory is useful for identifyin...,backgroundgame theory is useful for identifyin...,classical



Shape of the dataset: (1138, 4)


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

# turn the cleaned abstracts into tf idf features
tfidf = TfidfVectorizer(stop_words="english")

X = tfidf.fit_transform(df["Clean_Abstract"])
y = df["Label"]

print("Shape of tf idF data:", X.shape)
print("Number of labels:", len(y))

Shape of tf idF data: (1138, 8268)
Number of labels: 1138


In [3]:
# check a few of the words used as features
print("\nSome of the features:")
print(tfidf.get_feature_names_out()[:20])


Some of the features:
['aa' 'aaa' 'abandoned' 'abandonment' 'abar' 'abbreviation' 'abct'
 'ability' 'able' 'abm' 'abnormal' 'abo' 'abortive' 'abroad' 'abrupt'
 'absence' 'absent' 'absenteeism' 'absolut' 'absolute']


In [4]:
from sklearn.model_selection import train_test_split

# split the data into training and testing sets
# stratify keeps the class proportions similar in both sets
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)

print("train data:", X_train.shape)
print("Test Data:", X_test.shape)

# check that both sets still have the two classes
print("\nTraining labels:")
print(y_train.value_counts())

print("\nTesting labels:")
print(y_test.value_counts())

train data: (910, 8268)
Test Data: (228, 8268)

Training labels:
Label
classical    720
ai           190
Name: count, dtype: int64

Testing labels:
Label
classical    180
ai            48
Name: count, dtype: int64


In [5]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# build the baseline model
baseline_model = MultinomialNB()

baseline_model.fit(X_train, y_train)

# predictions for both training and testing data
y_train_pred = baseline_model.predict(X_train)
y_test_pred = baseline_model.predict(X_test)

# training performance
print("Training accuracy:", accuracy_score(y_train, y_train_pred))

print("\nTraining classification report:")
print(classification_report(y_train, y_train_pred))

print("\nTraining confusion matrix:")
print(confusion_matrix(y_train, y_train_pred))

# testing performance
print("\nTest accuracy:", accuracy_score(y_test, y_test_pred))

print("\nTest classification report:")
print(classification_report(y_test, y_test_pred))

print("\nTest confusion matrix:")
print(confusion_matrix(y_test, y_test_pred))

Training accuracy: 0.8098901098901099

Training classification report:
              precision    recall  f1-score   support

          ai       1.00      0.09      0.16       190
   classical       0.81      1.00      0.89       720

    accuracy                           0.81       910
   macro avg       0.90      0.54      0.53       910
weighted avg       0.85      0.81      0.74       910


Training confusion matrix:
[[ 17 173]
 [  0 720]]

Test accuracy: 0.793859649122807

Test classification report:
              precision    recall  f1-score   support

          ai       1.00      0.02      0.04        48
   classical       0.79      1.00      0.88       180

    accuracy                           0.79       228
   macro avg       0.90      0.51      0.46       228
weighted avg       0.84      0.79      0.71       228


Test confusion matrix:
[[  1  47]
 [  0 180]]


The baseline model achieved 80.99% training accuracy and 79.82% testing accuracy which shows similar performance on training and unseen data. However the model struggled to identify AI abstracts while performing very well on the Classical class.

# Question 2

2.1 train at least four machine learning models With the goal Of Obtaining
better predictive performance than the baseline model.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# create the four new models
logistic_model = LogisticRegression(max_iter=1000)
svm_model = LinearSVC()
knn_model = KNeighborsClassifier()
random_forest_model = RandomForestClassifier(random_state=42)

# train the models
logistic_model.fit(X_train, y_train)
svm_model.fit(X_train, y_train)
knn_model.fit(X_train, y_train)
random_forest_model.fit(X_train, y_train)

print("All four models have been trained.")

All four models have been trained.


In [7]:
# make predictions on the test data
logistic_pred = logistic_model.predict(X_test)
svm_pred = svm_model.predict(X_test)
knn_pred = knn_model.predict(X_test)
random_forest_pred = random_forest_model.predict(X_test)

# show the accuracy of each model
print("Logistic Regression accuracy:", accuracy_score(y_test, logistic_pred))
print("SVM accuracy:", accuracy_score(y_test, svm_pred))
print("KNN accuracy:", accuracy_score(y_test, knn_pred))
print("Random Forest accuracy:", accuracy_score(y_test, random_forest_pred))

Logistic Regression accuracy: 0.8464912280701754
SVM accuracy: 0.9035087719298246
KNN accuracy: 0.8728070175438597
Random Forest accuracy: 0.9035087719298246


In [8]:
print("Logistic Regression")
print(classification_report(y_test, logistic_pred))

print("SVM")
print(classification_report(y_test, svm_pred))

print("KNN")
print(classification_report(y_test, knn_pred))

print("Random Forest")
print(classification_report(y_test, random_forest_pred))

Logistic Regression
              precision    recall  f1-score   support

          ai       1.00      0.27      0.43        48
   classical       0.84      1.00      0.91       180

    accuracy                           0.85       228
   macro avg       0.92      0.64      0.67       228
weighted avg       0.87      0.85      0.81       228

SVM
              precision    recall  f1-score   support

          ai       0.93      0.58      0.72        48
   classical       0.90      0.99      0.94       180

    accuracy                           0.90       228
   macro avg       0.92      0.79      0.83       228
weighted avg       0.91      0.90      0.89       228

KNN
              precision    recall  f1-score   support

          ai       0.76      0.58      0.66        48
   classical       0.90      0.95      0.92       180

    accuracy                           0.87       228
   macro avg       0.83      0.77      0.79       228
weighted avg       0.87      0.87      0.87   

- SVM and Random Forest both achieved the highest accuracy at 90.35%.
- SVM had a slightly higher AI recall and AI F1-score than Random Forest.
- Logistic Regression and KNN performed below SVM and Random Forest.
**Based on these results, SVM is the strongest model overall.**

2.2. Besides the use of different models, provide one additional predictive performance enhancement approach of your
choice, showcasing with implementation how it improves the design performance

**SVM Performance Enhancement**
The c parameter controls the balance between correctly classifying the training data and allowing some classification errors. A higher c value makes the SVM place more importance on correctly classifying the training examples.
For this enhancement, the c value was increased to 2 to determine whether it would improve the model's predictive performance.

In [9]:
# try a higher C value for the SVM
improved_svm = LinearSVC(C=2)
improved_svm.fit(X_train, y_train)
improved_svm_pred = improved_svm.predict(X_test)

print("Improved SVM accuracy:", accuracy_score(y_test, improved_svm_pred))
print("\nClassification report:")
print(classification_report(y_test, improved_svm_pred))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, improved_svm_pred))

Improved SVM accuracy: 0.9122807017543859

Classification report:
              precision    recall  f1-score   support

          ai       0.94      0.62      0.75        48
   classical       0.91      0.99      0.95       180

    accuracy                           0.91       228
   macro avg       0.92      0.81      0.85       228
weighted avg       0.91      0.91      0.91       228


Confusion matrix:
[[ 30  18]
 [  2 178]]


In [10]:
import joblib

# save the trained SVM model
joblib.dump(improved_svm, "svm_model.pkl")

# save the TF-IDF vectorizer
joblib.dump(tfidf, "tfidf_vectorizer.pkl")
print("Model and vectorizer saved.")

Model and vectorizer saved.


In [11]:
%%writefile app.py

import streamlit as st
import joblib
import re
from nltk.stem import WordNetLemmatizer


# load the trained model and tf-idf vectorizer
model = joblib.load("svm_model.pkl")
tfidf = joblib.load("tfidf_vectorizer.pkl")

lemmatizer = WordNetLemmatizer()


# preprocess the new abstract
def clean_text(text):

    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words).strip()


# page configuration
st.set_page_config(page_title="Researchly Research Abstract Classifier",page_icon="📄")


# title
st.title("Researchly Research Abstract Classifier")

st.write("This app classifies research abstracts as either "
    "AI-related or classical research.")


# abstract input
abstract = st.text_area("Enter research abstract:",height=200)

# classification
if st.button("Classify Abstract"):

    if abstract.strip():
        # preprocess the abstract
        cleaned_text = clean_text(abstract)

        # convert the abstract to tf-idf
        text_features = tfidf.transform([cleaned_text])

        # make the prediction
        prediction = model.predict(text_features)[0]

        # make the label lowercase
        prediction = str(prediction).lower()

        st.subheader("Prediction")

        if prediction == "ai":
            st.success("The abstract is classified as: artificial intelligence")

        else:
            st.success("The abstract is classified as: classical research")
    else:
        st.warning("Please enter a research abstract.")

Overwriting app.py


In [12]:
import subprocess
import sys
import time

process = subprocess.Popen([sys.executable, "-m", "streamlit", "run","app.py", "--server.headless=true"  ],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)

time.sleep(3)

print("Streamlit application started.")
print("Open: http://localhost:8501")

Streamlit application started.
Open: http://localhost:8501
